# 04 — AIA Baseline Protocol and Year-Holdout Folds

**Purpose:** define the formal 2010–2016 AIA image baseline protocol before literature-style CNN benchmarking.

This notebook does **not** train a model. It creates protocol files for later notebooks.

## Fixed decisions

- Dataset: `baseline_2010_2016_AR_SPECIFIC_manifest.csv`
- Input tensor key: `x`
- Input tensor shape: `(512, 512, 6)`
- Channels: `aia94`, `aia131`, `aia171`, `aia193`, `aia211`, `aia335`
- Label: `label_48h_final`
- Horizon: 48-hour M/X flare prediction
- Split style: chronological rolling year-holdout
- Threshold rule: select threshold on validation by max TSS, then apply unchanged to test
- Embedded NPZ `y` is ignored for formal training

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

ROOT = Path.home() / "solar_flare_aia"

MANIFEST = ROOT / "training/final_metadata/baseline_2010_2016_AR_SPECIFIC_manifest.csv"

METRICS_DIR = ROOT / "results/metrics"
METRICS_DIR.mkdir(parents=True, exist_ok=True)

EXPERIMENT_NAME = "aia_baseline_2010_2016_protocol"

YEAR_COUNTS_CSV = METRICS_DIR / f"{EXPERIMENT_NAME}_year_counts.csv"
FOLD_SUMMARY_CSV = METRICS_DIR / f"{EXPERIMENT_NAME}_fold_summary.csv"
FOLD_ASSIGNMENTS_CSV = METRICS_DIR / f"{EXPERIMENT_NAME}_fold_assignments.csv"
PROTOCOL_JSON = METRICS_DIR / f"{EXPERIMENT_NAME}.json"
README_MD = METRICS_DIR / f"{EXPERIMENT_NAME}_readable_summary.md"

print("Project root:", ROOT)
print("Manifest:", MANIFEST)
print("Outputs:", METRICS_DIR)

Project root: /home/abmoses2000/solar_flare_aia
Manifest: /home/abmoses2000/solar_flare_aia/training/final_metadata/baseline_2010_2016_AR_SPECIFIC_manifest.csv
Outputs: /home/abmoses2000/solar_flare_aia/results/metrics


## 1. Load manifest and verify required fields

In [2]:
required_columns = [
    "gcp_path",
    "sample_id",
    "T_REC_dt",
    "HARPNUM",
    "NOAA_AR_clean",
    "label_48h_final",
    "year",
]

df = pd.read_csv(MANIFEST, low_memory=False)

print("Rows:", len(df))
print("Columns:", df.columns.tolist())

missing_cols = [c for c in required_columns if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

missing_report = {c: int(df[c].isna().sum()) for c in required_columns}

display(Markdown("### Missing-value check"))
display(pd.DataFrame([missing_report]).T.rename(columns={0: "missing_count"}))

if missing_report["gcp_path"] != 0:
    raise ValueError("gcp_path has missing values.")
if missing_report["sample_id"] != 0:
    raise ValueError("sample_id has missing values.")
if missing_report["label_48h_final"] != 0:
    raise ValueError("label_48h_final has missing values.")

df["year"] = df["year"].astype(int)
df["label_48h_final"] = df["label_48h_final"].astype(int)

print("Manifest loaded and core fields verified.")

Rows: 68010
Columns: ['gcp_path', 'file', 'sample_id', 'T_REC_dt', 'HARPNUM', 'NOAA_AR_clean', 'label_48h_global_old', 'label_48h_ar_specific', 'label_48h_final', 'label_48h', 'y', 'local_path', 'year', 'used_timestamp', 'used_s3_path', 'manifest_label_before_repair']


### Missing-value check

,missing_count
gcp_path,0
sample_id,0
T_REC_dt,0
HARPNUM,0
NOAA_AR_clean,0
label_48h_final,0
year,0


Manifest loaded and core fields verified.


## 2. Year-level class distribution

In [3]:
year_counts = (
    df.groupby("year")
      .agg(
          rows=("sample_id", "count"),
          positives=("label_48h_final", "sum"),
          negatives=("label_48h_final", lambda s: int((s == 0).sum())),
          unique_harpnums=("HARPNUM", "nunique"),
          unique_noaa_ars=("NOAA_AR_clean", "nunique"),
      )
      .reset_index()
)

year_counts["positive_rate"] = year_counts["positives"] / year_counts["rows"]
year_counts.to_csv(YEAR_COUNTS_CSV, index=False)

display(Markdown("### Baseline 2010–2016 year counts"))
display(year_counts)

print("Saved:", YEAR_COUNTS_CSV)

### Baseline 2010–2016 year counts

,year,rows,positives,negatives,unique_harpnums,unique_noaa_ars,positive_rate
0,2010,3306,29,3277,58,57,0.008772
1,2011,11142,404,10738,185,183,0.036259
2,2012,10815,305,10510,192,189,0.028202
3,2013,13066,492,12574,230,222,0.037655
4,2014,11627,629,10998,190,187,0.054098
5,2015,11236,531,10705,186,180,0.047259
6,2016,6818,8,6810,122,120,0.001173


Saved: /home/abmoses2000/solar_flare_aia/results/metrics/aia_baseline_2010_2016_protocol_year_counts.csv


## 3. Formal chronological year-holdout protocol

### Selected protocol: rolling-origin chronological folds

For each fold:

- `test_year = Y`
- `val_year = Y - 1`
- `train_years = all years before val_year`
- Threshold is selected on validation only by maximum TSS.
- The selected threshold is applied unchanged to the test year.
- No future year is used for training or threshold selection.

Candidate folds:

- Fold 2012: train 2010, validate 2011, test 2012
- Fold 2013: train 2010–2011, validate 2012, test 2013
- Fold 2014: train 2010–2012, validate 2013, test 2014
- Fold 2015: train 2010–2013, validate 2014, test 2015
- Fold 2016: train 2010–2014, validate 2015, test 2016

In [4]:
BASELINE_YEARS = list(range(2010, 2017))
TEST_YEARS = list(range(2012, 2017))

folds = []

for test_year in TEST_YEARS:
    val_year = test_year - 1
    train_years = [y for y in BASELINE_YEARS if y < val_year]

    train_df = df[df["year"].isin(train_years)]
    val_df = df[df["year"] == val_year]
    test_df = df[df["year"] == test_year]

    fold = {
        "fold_id": f"test_{test_year}",
        "train_years": train_years,
        "val_year": val_year,
        "test_year": test_year,
        "train_rows": int(len(train_df)),
        "train_pos": int(train_df["label_48h_final"].sum()),
        "train_neg": int((train_df["label_48h_final"] == 0).sum()),
        "train_pos_rate": float(train_df["label_48h_final"].mean()) if len(train_df) else np.nan,
        "val_rows": int(len(val_df)),
        "val_pos": int(val_df["label_48h_final"].sum()),
        "val_neg": int((val_df["label_48h_final"] == 0).sum()),
        "val_pos_rate": float(val_df["label_48h_final"].mean()) if len(val_df) else np.nan,
        "test_rows": int(len(test_df)),
        "test_pos": int(test_df["label_48h_final"].sum()),
        "test_neg": int((test_df["label_48h_final"] == 0).sum()),
        "test_pos_rate": float(test_df["label_48h_final"].mean()) if len(test_df) else np.nan,
    }

    fold["usable_for_full_benchmark"] = bool(
        fold["train_pos"] >= 50 and fold["val_pos"] >= 10 and fold["test_pos"] >= 10
    )

    if fold["test_pos"] < 10:
        fold["notes"] = "Very low positive count in test year; report cautiously or treat as stress fold."
    elif fold["val_pos"] < 10:
        fold["notes"] = "Very low positive count in validation year; threshold selection may be unstable."
    elif fold["train_pos"] < 50:
        fold["notes"] = "Low training positive count; may be unstable for deep CNN training."
    else:
        fold["notes"] = "Candidate fold for formal benchmark."

    folds.append(fold)

fold_summary = pd.DataFrame(folds)
fold_summary.to_csv(FOLD_SUMMARY_CSV, index=False)

display(Markdown("### Chronological fold summary"))
display(fold_summary)

print("Saved:", FOLD_SUMMARY_CSV)

### Chronological fold summary

,fold_id,train_years,val_year,test_year,train_rows,train_pos,train_neg,train_pos_rate,val_rows,val_pos,val_neg,val_pos_rate,test_rows,test_pos,test_neg,test_pos_rate,usable_for_full_benchmark,notes
0,test_2012,[2010],2011,2012,3306,29,3277,0.008772,11142,404,10738,0.036259,10815,305,10510,0.028202,False,Low training positive count; may be unstable f...
1,test_2013,"[2010, 2011]",2012,2013,14448,433,14015,0.029970,10815,305,10510,0.028202,13066,492,12574,0.037655,True,Candidate fold for formal benchmark.
2,test_2014,"[2010, 2011, 2012]",2013,2014,25263,738,24525,0.029213,13066,492,12574,0.037655,11627,629,10998,0.054098,True,Candidate fold for formal benchmark.
3,test_2015,"[2010, 2011, 2012, 2013]",2014,2015,38329,1230,37099,0.032091,11627,629,10998,0.054098,11236,531,10705,0.047259,True,Candidate fold for formal benchmark.
4,test_2016,"[2010, 2011, 2012, 2013, 2014]",2015,2016,49956,1859,48097,0.037213,11236,531,10705,0.047259,6818,8,6810,0.001173,False,Very low positive count in test year; report c...


Saved: /home/abmoses2000/solar_flare_aia/results/metrics/aia_baseline_2010_2016_protocol_fold_summary.csv


## 4. Create sample-level fold assignment file

In [5]:
assignment_rows = []

for fold in folds:
    fold_id = fold["fold_id"]

    for split_name, years in [
        ("train", fold["train_years"]),
        ("val", [fold["val_year"]]),
        ("test", [fold["test_year"]]),
    ]:
        split_df = df[df["year"].isin(years)][
            [
                "gcp_path",
                "sample_id",
                "T_REC_dt",
                "HARPNUM",
                "NOAA_AR_clean",
                "label_48h_final",
                "year",
            ]
        ].copy()

        split_df.insert(0, "fold_id", fold_id)
        split_df.insert(1, "split", split_name)
        assignment_rows.append(split_df)

fold_assignments = pd.concat(assignment_rows, ignore_index=True)
fold_assignments.to_csv(FOLD_ASSIGNMENTS_CSV, index=False)

display(Markdown("### Fold assignment preview"))
display(fold_assignments.head())

display(Markdown("### Assignment row counts"))
display(fold_assignments.groupby(["fold_id", "split"])["sample_id"].count().unstack())

print("Saved:", FOLD_ASSIGNMENTS_CSV)
print("Rows:", len(fold_assignments))

### Fold assignment preview

,fold_id,split,gcp_path,sample_id,T_REC_dt,HARPNUM,NOAA_AR_clean,label_48h_final,year
0,test_2012,train,gs://suryabench-sharp-pipeline-bamidele/sample...,20100521_1636_HARP26_NOAA11072,2010-05-21 16:36:00,26,11072,0,2010
1,test_2012,train,gs://suryabench-sharp-pipeline-bamidele/sample...,20100521_1812_HARP26_NOAA11072,2010-05-21 18:12:00,26,11072,0,2010
2,test_2012,train,gs://suryabench-sharp-pipeline-bamidele/sample...,20100521_1948_HARP26_NOAA11072,2010-05-21 19:48:00,26,11072,0,2010
3,test_2012,train,gs://suryabench-sharp-pipeline-bamidele/sample...,20100521_2124_HARP26_NOAA11072,2010-05-21 21:24:00,26,11072,0,2010
4,test_2012,train,gs://suryabench-sharp-pipeline-bamidele/sample...,20100521_2300_HARP26_NOAA11072,2010-05-21 23:00:00,26,11072,0,2010


### Assignment row counts

split,test,train,val
fold_id,,,
test_2012,10815,3306,11142
test_2013,13066,14448,10815
test_2014,11627,25263,13066
test_2015,11236,38329,11627
test_2016,6818,49956,11236


Saved: /home/abmoses2000/solar_flare_aia/results/metrics/aia_baseline_2010_2016_protocol_fold_assignments.csv
Rows: 242750


## 5. Metric, threshold, and class-imbalance rules

In [8]:
metric_protocol = {
    "threshold_independent_metrics": [
        "ROC-AUC",
        "PR-AUC",
        "Brier score",
    ],
    "threshold_dependent_metrics": [
        "TSS",
        "HSS",
        "Accuracy",
        "Precision",
        "Recall",
        "Specificity",
        "F1",
        "Confusion matrix: TP, TN, FP, FN",
    ],
    "threshold_rules": {
        "fixed_threshold": 0.5,
        "selected_threshold": "Choose threshold on validation split by maximum TSS; apply unchanged to test split.",
        "no_test_threshold_tuning": True,
    },
    "class_imbalance_rules": {
        "primary_reporting": "Use realistic class distribution for formal benchmark.",
        "allowed_training_strategies": [
            "BCEWithLogitsLoss with pos_weight computed from training split",
            "WeightedRandomSampler only if explicitly documented",
        ],
        "balanced_subset_use": "Allowed only for sanity checks, not final benchmark claims.",
    },
}

threshold_independent_df = pd.DataFrame({
    "metric_type": "Threshold-independent",
    "metric": metric_protocol["threshold_independent_metrics"],
})

threshold_dependent_df = pd.DataFrame({
    "metric_type": "Threshold-dependent",
    "metric": metric_protocol["threshold_dependent_metrics"],
})

metric_table = pd.concat(
    [threshold_independent_df, threshold_dependent_df],
    ignore_index=True
)

display(Markdown("### Metric protocol"))
display(metric_table)

display(Markdown("### Threshold rule"))
display(pd.DataFrame([
    {
        "fixed_threshold": metric_protocol["threshold_rules"]["fixed_threshold"],
        "selected_threshold_rule": metric_protocol["threshold_rules"]["selected_threshold"],
        "test_threshold_tuning_allowed": not metric_protocol["threshold_rules"]["no_test_threshold_tuning"],
    }
]))

display(Markdown("### Class imbalance rule"))
display(pd.DataFrame([
    {
        "primary_reporting": metric_protocol["class_imbalance_rules"]["primary_reporting"],
        "balanced_subset_use": metric_protocol["class_imbalance_rules"]["balanced_subset_use"],
    }
]))

metric_protocol

### Metric protocol

,metric_type,metric
0,Threshold-independent,ROC-AUC
1,Threshold-independent,PR-AUC
2,Threshold-independent,Brier score
3,Threshold-dependent,TSS
4,Threshold-dependent,HSS
5,Threshold-dependent,Accuracy
6,Threshold-dependent,Precision
7,Threshold-dependent,Recall
8,Threshold-dependent,Specificity
9,Threshold-dependent,F1


### Threshold rule

,fixed_threshold,selected_threshold_rule,test_threshold_tuning_allowed
0,0.5,Choose threshold on validation split by maximu...,False


### Class imbalance rule

,primary_reporting,balanced_subset_use
0,Use realistic class distribution for formal be...,"Allowed only for sanity checks, not final benc..."


{'threshold_independent_metrics': ['ROC-AUC', 'PR-AUC', 'Brier score'],
 'threshold_dependent_metrics': ['TSS',
  'HSS',
  'Accuracy',
  'Precision',
  'Recall',
  'Specificity',
  'F1',
  'Confusion matrix: TP, TN, FP, FN'],
 'threshold_rules': {'fixed_threshold': 0.5,
  'selected_threshold': 'Choose threshold on validation split by maximum TSS; apply unchanged to test split.',
  'no_test_threshold_tuning': True},
 'class_imbalance_rules': {'primary_reporting': 'Use realistic class distribution for formal benchmark.',
  'allowed_training_strategies': ['BCEWithLogitsLoss with pos_weight computed from training split',
   'WeightedRandomSampler only if explicitly documented'],
  'balanced_subset_use': 'Allowed only for sanity checks, not final benchmark claims.'}}

## 6. Save protocol JSON and readable summary

In [7]:
protocol = {
    "experiment_name": EXPERIMENT_NAME,
    "purpose": "Formal AIA 2010-2016 baseline protocol and chronological year-holdout fold definition.",
    "dataset": {
        "manifest": str(MANIFEST),
        "years": BASELINE_YEARS,
        "input_tensor_key": "x",
        "input_shape_hwc": [512, 512, 6],
        "channels": ["aia94", "aia131", "aia171", "aia193", "aia211", "aia335"],
        "label_column": "label_48h_final",
        "ignored_npz_label": "y",
        "task": "M/X flare prediction within 48 hours for the same active region.",
    },
    "split_protocol": {
        "name": "rolling_origin_chronological_year_holdout",
        "description": (
            "For test year Y, validation year is Y-1 and training years are all years before Y-1. "
            "No future data are used for model training or threshold selection."
        ),
        "test_years": TEST_YEARS,
        "folds": folds,
    },
    "metric_protocol": metric_protocol,
    "outputs": {
        "year_counts_csv": str(YEAR_COUNTS_CSV),
        "fold_summary_csv": str(FOLD_SUMMARY_CSV),
        "fold_assignments_csv": str(FOLD_ASSIGNMENTS_CSV),
        "protocol_json": str(PROTOCOL_JSON),
        "readable_summary_md": str(README_MD),
    },
    "status": "Protocol only. No model training performed in this notebook.",
}

PROTOCOL_JSON.write_text(json.dumps(protocol, indent=2))

summary_lines = []
summary_lines.append("# AIA Baseline 2010–2016 Protocol Summary\n")
summary_lines.append("## Dataset\n")
summary_lines.append(f"- Manifest: `{MANIFEST}`")
summary_lines.append("- Input tensor: `x`, shape `(512, 512, 6)`")
summary_lines.append("- Channels: `aia94`, `aia131`, `aia171`, `aia193`, `aia211`, `aia335`")
summary_lines.append("- Label: `label_48h_final`")
summary_lines.append("- Embedded NPZ `y` is ignored for formal training.")
summary_lines.append("- Task: same-active-region M/X flare prediction within 48 hours.\n")

summary_lines.append("## Split protocol\n")
summary_lines.append("- Rolling-origin chronological year-holdout.")
summary_lines.append("- For test year `Y`, validation year is `Y-1`; training years are all years before `Y-1`.")
summary_lines.append("- Threshold is selected on validation by maximum TSS and applied unchanged to test.\n")

summary_lines.append("## Candidate folds\n")
for _, row in fold_summary.iterrows():
    summary_lines.append(
        f"- `{row['fold_id']}`: train={row['train_years']}, "
        f"val={int(row['val_year'])}, test={int(row['test_year'])}, "
        f"train_pos={int(row['train_pos'])}, val_pos={int(row['val_pos'])}, test_pos={int(row['test_pos'])}, "
        f"usable={bool(row['usable_for_full_benchmark'])}. {row['notes']}"
    )

summary_lines.append("\n## Outputs\n")
summary_lines.append(f"- Year counts: `{YEAR_COUNTS_CSV}`")
summary_lines.append(f"- Fold summary: `{FOLD_SUMMARY_CSV}`")
summary_lines.append(f"- Fold assignments: `{FOLD_ASSIGNMENTS_CSV}`")
summary_lines.append(f"- Protocol JSON: `{PROTOCOL_JSON}`")

README_MD.write_text("\n".join(summary_lines) + "\n")

print("Saved:", PROTOCOL_JSON)
print("Saved:", README_MD)

display(Markdown(README_MD.read_text()))

Saved: /home/abmoses2000/solar_flare_aia/results/metrics/aia_baseline_2010_2016_protocol.json
Saved: /home/abmoses2000/solar_flare_aia/results/metrics/aia_baseline_2010_2016_protocol_readable_summary.md


# AIA Baseline 2010–2016 Protocol Summary

## Dataset

- Manifest: `/home/abmoses2000/solar_flare_aia/training/final_metadata/baseline_2010_2016_AR_SPECIFIC_manifest.csv`
- Input tensor: `x`, shape `(512, 512, 6)`
- Channels: `aia94`, `aia131`, `aia171`, `aia193`, `aia211`, `aia335`
- Label: `label_48h_final`
- Embedded NPZ `y` is ignored for formal training.
- Task: same-active-region M/X flare prediction within 48 hours.

## Split protocol

- Rolling-origin chronological year-holdout.
- For test year `Y`, validation year is `Y-1`; training years are all years before `Y-1`.
- Threshold is selected on validation by maximum TSS and applied unchanged to test.

## Candidate folds

- `test_2012`: train=[2010], val=2011, test=2012, train_pos=29, val_pos=404, test_pos=305, usable=False. Low training positive count; may be unstable for deep CNN training.
- `test_2013`: train=[2010, 2011], val=2012, test=2013, train_pos=433, val_pos=305, test_pos=492, usable=True. Candidate fold for formal benchmark.
- `test_2014`: train=[2010, 2011, 2012], val=2013, test=2014, train_pos=738, val_pos=492, test_pos=629, usable=True. Candidate fold for formal benchmark.
- `test_2015`: train=[2010, 2011, 2012, 2013], val=2014, test=2015, train_pos=1230, val_pos=629, test_pos=531, usable=True. Candidate fold for formal benchmark.
- `test_2016`: train=[2010, 2011, 2012, 2013, 2014], val=2015, test=2016, train_pos=1859, val_pos=531, test_pos=8, usable=False. Very low positive count in test year; report cautiously or treat as stress fold.

## Outputs

- Year counts: `/home/abmoses2000/solar_flare_aia/results/metrics/aia_baseline_2010_2016_protocol_year_counts.csv`
- Fold summary: `/home/abmoses2000/solar_flare_aia/results/metrics/aia_baseline_2010_2016_protocol_fold_summary.csv`
- Fold assignments: `/home/abmoses2000/solar_flare_aia/results/metrics/aia_baseline_2010_2016_protocol_fold_assignments.csv`
- Protocol JSON: `/home/abmoses2000/solar_flare_aia/results/metrics/aia_baseline_2010_2016_protocol.json`


## 7. Next notebook after this protocol

After saving and committing this protocol notebook, the next notebook should train the first formal CNN baseline:

```text
05_aia_literature_cnn_baseline_benchmark.ipynb
```

Recommended first benchmark run:

- Architecture: AlexNet-style six-channel CNN
- Training: realistic class imbalance
- Loss: `BCEWithLogitsLoss` with `pos_weight` from training split
- First fold: likely `test_2015`, because it has a sensible train/validation/test chronology
- Threshold: select on validation by maximum TSS
- Report test metrics at both `0.5` and validation-selected threshold

Only after the first fold works should we expand to more folds and stronger architectures.